In [2]:
# Install PDF reader
!pip install pymupdf python-docx -q

import fitz  # pymupdf
from docx import Document as DocxDocument
import os
import numpy as np
def load_document(filepath):
    """
    Load text from PDF, DOCX, or TXT.
    Returns full text string.
    """
    ext = Path(filepath).suffix.lower()
    
    if ext == ".pdf":
        doc  = fitz.open(filepath)
        text = ""
        for page in doc:
            text += page.get_text()
        doc.close()
        return text
    
    elif ext == ".docx":
        doc   = DocxDocument(filepath)
        paras = [p.text for p in doc.paragraphs if p.text.strip()]
        return "\n".join(paras)
    
    elif ext == ".txt":
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    
    else:
        print(f"Skipping unknown format: {filepath}")
        return ""


def chunk_text(text, doc_name, chunk_size=256, overlap=50):
    """
    Split text into overlapping chunks.
    chunk_size and overlap in words.
    Returns list of dicts with text and metadata.
    """
    words  = text.split()
    chunks = []
    start  = 0
    idx    = 0
    
    while start < len(words):
        end        = min(start + chunk_size, len(words))
        chunk_text = " ".join(words[start:end])
        
        chunks.append({
            "chunk_id"  : f"{doc_name}_chunk_{idx}",
            "doc_name"  : doc_name,
            "text"      : chunk_text,
            "word_start": start,
            "word_end"  : end
        })
        
        if end == len(words):
            break
        
        start += chunk_size - overlap
        idx   += 1
    
    return chunks


def load_all_documents(doc_dir):
    """
    Load and chunk all documents in directory.
    """
    all_chunks = []
    doc_files  = [
        f for f in os.listdir(doc_dir)
        if f.endswith((".pdf", ".docx", ".txt"))
    ]
    
    print(f"Found {len(doc_files)} documents")
    
    for fname in doc_files:
        fpath    = os.path.join(doc_dir, fname)
        doc_name = Path(fname).stem
        
        print(f"Loading: {fname}...")
        text = load_document(fpath)
        
        if not text.strip():
            print(f"  WARNING: Empty document — {fname}")
            continue
        
        chunks = chunk_text(text, doc_name)
        all_chunks.extend(chunks)
        print(f"  {len(chunks)} chunks from {len(text.split())} words")
    
    print(f"\nTotal chunks: {len(all_chunks)}")
    return all_chunks


# Run
from pathlib import Path

doc_dir    = "/kaggle/input/datasets/sauraviitjapdcl/duomind-sshrag-mahalanobis"  # adjust path
all_chunks = load_all_documents(doc_dir)

# Convert to list of texts for embedding
chunk_texts = [c["text"] for c in all_chunks]
chunk_ids   = [c["chunk_id"] for c in all_chunks]

print(f"\nSample chunk:")
print(f"ID  : {chunk_ids[0]}")
print(f"Text: {chunk_texts[0][:200]}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 70.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.0 MB/s eta 0:00:00
Found 13 documents
Loading: 24062022_Guidelines_on_Cyber_Security_in_Power_Sector_2021.pdf...
  40 chunks from 8245 words
Loading: NCIIPC_Guidelines_V2.pdf...
  72 chunks from 14703 words
Loading: CERT-In_Directions.docx...
  9 chunks from 1845 words
Loading: DPDP ACT.pdf...
  52 chunks from 10685 words
Loading: e-Governance Policy Initiatives.pdf...
  213 chunks from 43724 words
Loading: Electricity Act 2003.pdf...
  200 chunks from 41211 words
Loading: guidelinesgovtentities.pdf...
  74 chunks from 15283 words
Loading: Cyber Security Framework in Banks.docx...
  33 chunks from 6732 words
Loading: ISO_IEC-270012022-ed.3.pdf...
  35 chunks from 7195 words
Loading: nist cyber security framework.docx...
  38 chunks from 7834 words
Loading: OECD-LEGAL-0479-en.docx...
  20 chunks from 4154 words
Loading: NCIIPC-Rules-noti

In [3]:
import pandas as pd

# Load 340Q
qa_df = pd.read_csv("/kaggle/input/datasets/sauraviitjapdcl/ssh-rag-code/ssh_rag_groundtruth.csv")
print(f"Questions: {len(qa_df)}")
print(qa_df.head(3))

def build_gold_labels(qa_df, chunk_texts):
    """
    Match ground truth answer to chunk containing it.
    Returns dict {question: gold_chunk_idx}
    """
    gold_labels  = {}
    not_found    = []
    
    for _, row in qa_df.iterrows():
        question = row["question"]
        answer   = str(row["ground_truth_answer"]).lower().strip()
        
        found = False
        for idx, chunk in enumerate(chunk_texts):
            if answer in chunk.lower():
                gold_labels[question] = idx
                found = True
                break
        
        if not found:
            not_found.append(question)
    
    print(f"Gold labels found : {len(gold_labels)}/{len(qa_df)}")
    print(f"Not found         : {len(not_found)}")
    if not_found:
        print(f"First 3 not found :")
        for q in not_found[:3]:
            print(f"  {q}")
    
    return gold_labels, not_found

gold_labels, not_found = build_gold_labels(qa_df, chunk_texts)

Questions: 340
   id document_source                               question  \
0   1   DPDP Act 2023  What is personal data under DPDP Act?   
1   2   DPDP Act 2023               Who is a data fiduciary?   
2   3   DPDP Act 2023               Who is a data principal?   

                                 ground_truth_answer  
0  Any data about an individual who is identifiab...  
1  Any person who determines the purpose and mean...  
2  The individual to whom the personal data relates.  
Gold labels found : 0/340
Not found         : 340
First 3 not found :
  What is personal data under DPDP Act?
  Who is a data fiduciary?
  Who is a data principal?


In [5]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
import torch
import numpy as np

print("Loading BGE...")
model     = SentenceTransformer("BAAI/bge-base-en-v1.5", device="cuda")
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")
print(f"BGE loaded on: {next(model.parameters()).device}")

Loading BGE...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BGE loaded on: cuda:0


In [6]:
# Semantic gold label creation
# Embed ground truth answers
# Find most similar chunk by cosine similarity
# Much better for paraphrased answers

print("Embedding 340 ground truth answers...")
answer_texts = qa_df["ground_truth_answer"].tolist()
answer_embeddings = model.encode(
    answer_texts,
    batch_size           = 64,
    show_progress_bar    = True,
    normalize_embeddings = True,
    convert_to_numpy     = True
)
print(f"Answer embeddings: {answer_embeddings.shape}")

print("\nEmbedding 901 DuoMind chunks...")
duomind_embeddings = model.encode(
    chunk_texts,
    batch_size           = 64,
    show_progress_bar    = True,
    normalize_embeddings = True,
    convert_to_numpy     = True
)
print(f"Chunk embeddings: {duomind_embeddings.shape}")

# Find best matching chunk for each answer
print("\nBuilding semantic gold labels...")
gold_labels    = {}
gold_scores    = []
questions_list = qa_df["question"].tolist()

for i, (question, answer_emb) in enumerate(
    zip(questions_list, answer_embeddings)
):
    # Cosine similarity between answer and all chunks
    scores   = duomind_embeddings @ answer_emb
    best_idx = int(np.argmax(scores))
    best_score = float(scores[best_idx])
    
    gold_labels[question] = best_idx
    gold_scores.append(best_score)

print(f"Gold labels built: {len(gold_labels)}")
print(f"Avg similarity   : {np.mean(gold_scores):.4f}")
print(f"Min similarity   : {np.min(gold_scores):.4f}")
print(f"Max similarity   : {np.max(gold_scores):.4f}")

# Sanity check — first 3
print("\n=== SANITY CHECK ===")
for i in range(3):
    q     = questions_list[i]
    idx   = gold_labels[q]
    score = gold_scores[i]
    print(f"\nQ: {q}")
    print(f"A: {answer_texts[i]}")
    print(f"Best chunk (score={score:.4f}):")
    print(f"  {chunk_texts[idx][:200]}")

Embedding 340 ground truth answers...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Answer embeddings: (340, 768)

Embedding 901 DuoMind chunks...


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Chunk embeddings: (901, 768)

Building semantic gold labels...
Gold labels built: 200
Avg similarity   : 0.6923
Min similarity   : 0.4933
Max similarity   : 0.8358

=== SANITY CHECK ===

Q: What is personal data under DPDP Act?
A: Any data about an individual who is identifiable by or in relation to such data.
Best chunk (score=0.6997):
  a single point of contact to enable a Data Principal to give, manage, review and withdraw her consent through an accessible, transparent and interoperable platform; (h) “data” means a representation o

Q: Who is a data fiduciary?
A: Any person who determines the purpose and means of processing personal data.
Best chunk (score=0.7249):
  a single point of contact to enable a Data Principal to give, manage, review and withdraw her consent through an accessible, transparent and interoperable platform; (h) “data” means a representation o

Q: Who is a data principal?
A: The individual to whom the personal data relates.
Best chunk (score=0.7524):
  a single

In [8]:
# Check sizes
print(f"qa_df rows         : {len(qa_df)}")
print(f"questions_list len : {len(questions_list)}")
print(f"answer_texts len   : {len(answer_texts)}")
print(f"answer_embeddings  : {answer_embeddings.shape}")
print(f"gold_labels built  : {len(gold_labels)}")

# Fix — use index directly, not zip
gold_labels  = {}
gold_scores  = []

for i in range(len(qa_df)):
    question   = qa_df.iloc[i]["question"]
    answer_emb = answer_embeddings[i]

    scores     = duomind_embeddings @ answer_emb
    best_idx   = int(np.argmax(scores))
    best_score = float(scores[best_idx])

    gold_labels[question] = best_idx
    gold_scores.append(best_score)

print(f"\nGold labels built: {len(gold_labels)}")
print(f"Avg similarity   : {np.mean(gold_scores):.4f}")

# Better sanity check — show diverse examples
print("\n=== SANITY CHECK ===")
for i in [0, 50, 100, 150, 200]:
    if i >= len(qa_df):
        break
    q     = qa_df.iloc[i]["question"]
    a     = qa_df.iloc[i]["ground_truth_answer"]
    idx   = gold_labels[q]
    score = gold_scores[i]
    print(f"\nQ    : {q}")
    print(f"A    : {a[:100]}")
    print(f"Chunk: {chunk_texts[idx][:150]}")
    print(f"Score: {score:.4f}")
    print(f"Doc  : {chunk_ids[idx].split('_chunk_')[0]}")

qa_df rows         : 340
questions_list len : 340
answer_texts len   : 340
answer_embeddings  : (340, 768)
gold_labels built  : 200

Gold labels built: 200
Avg similarity   : 0.6923

=== SANITY CHECK ===

Q    : What is personal data under DPDP Act?
A    : Any data about an individual who is identifiable by or in relation to such data.
Chunk: a single point of contact to enable a Data Principal to give, manage, review and withdraw her consent through an accessible, transparent and interoper
Score: 0.6997
Doc  : DPDP ACT

Q    : What is a Sectoral CERT?
A    : A sector-specific CERT that handles incidents and advisories for a particular CII sector.
Chunk: of CERT-In www.cert-in.org.in and will be updated from time to time. When required by order/direction of CERT-In, for the purposes of cyber incident r
Score: 0.7102
Doc  : CERT-In_Directions

Q    : What happens if supplier security agreements are absent?
A    : Third-party breaches can compromise organizational data with no legal reco

In [9]:
print(f"Unique questions: {qa_df['question'].nunique()}")
print(f"Total questions : {len(qa_df)}")
duplicates = qa_df[qa_df.duplicated('question', keep=False)]
print(f"Duplicate rows  : {len(duplicates)}")
print(duplicates['question'].value_counts().head(5))

Unique questions: 200
Total questions : 340
Duplicate rows  : 175
question
How do CERT-In Directions and the RBI Cyber Security Framework complement each other for cyber resilience?    5
In what way can CERT-In reporting requirements be mapped to the NIST Cybersecurity Framework lifecycle?       5
To what extent do NIST CSF and OECD Guidelines together improve enterprise cyber governance?                  5
Why can Government IT resource policies support compliance with CERT-In Directions?                           5
How do RBI and NIST frameworks jointly strengthen cyber risk governance?                                      5
Name: count, dtype: int64


In [10]:
# Deduplicate keeping first occurrence
qa_df_clean = qa_df.drop_duplicates(subset='question', keep='first')
print(f"After dedup: {len(qa_df_clean)} unique questions")

# Rebuild with clean df
questions_list = qa_df_clean["question"].tolist()
answer_texts   = qa_df_clean["ground_truth_answer"].tolist()

# Re-embed answers
print("Re-embedding 200 unique answers...")
answer_embeddings = model.encode(
    answer_texts,
    batch_size           = 64,
    show_progress_bar    = True,
    normalize_embeddings = True,
    convert_to_numpy     = True
)

# Rebuild gold labels
gold_labels = {}
gold_scores = []

for i in range(len(qa_df_clean)):
    question   = qa_df_clean.iloc[i]["question"]
    answer_emb = answer_embeddings[i]
    scores     = duomind_embeddings @ answer_emb
    best_idx   = int(np.argmax(scores))
    best_score = float(scores[best_idx])
    gold_labels[question] = best_idx
    gold_scores.append(best_score)

print(f"Gold labels: {len(gold_labels)}/200")
print(f"Avg score  : {np.mean(gold_scores):.4f}")
print(f"High confidence (>0.75): {sum(1 for s in gold_scores if s > 0.75)}")
print(f"Medium (0.6-0.75)      : {sum(1 for s in gold_scores if 0.6 <= s <= 0.75)}")
print(f"Low (<0.6)             : {sum(1 for s in gold_scores if s < 0.6)}")

After dedup: 200 unique questions
Re-embedding 200 unique answers...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Gold labels: 200/200
Avg score  : 0.6799
High confidence (>0.75): 16
Medium (0.6-0.75)      : 175
Low (<0.6)             : 9


In [16]:
!pip install rank_bm25 sentence-transformers scikit-learn networkx numpy scipy -q

In [17]:
import os
import numpy as np
import networkx as nx
from collections import defaultdict
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, models
from sklearn.covariance import LedoitWolf
from sklearn.metrics.pairwise import euclidean_distances, cosine_distances
from transformers import AutoTokenizer
import torch

# Confirm GPU
print(f"Device : {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6GB


In [20]:
import numpy as np
import networkx as nx
from collections import defaultdict
from rank_bm25 import BM25Okapi
from sklearn.covariance import LedoitWolf
from sklearn.metrics.pairwise import euclidean_distances, cosine_distances
from transformers import AutoTokenizer
import types

# ── evaluate ─────────────────────────────────────────────
def evaluate(retrieved_dict, relevance, query_ids,
             top_k_list=[5,10], label=""):
    ap_scores = []
    recalls   = {k: [] for k in top_k_list}
    for q_id in query_ids:
        if q_id not in relevance or not relevance[q_id]:
            continue
        relevant = relevance[q_id]
        ranked   = [r[0] for r in retrieved_dict[q_id]]
        hits, ap = 0, 0.0
        for rank, idx in enumerate(ranked, 1):
            if idx in relevant:
                hits += 1
                ap   += hits / rank
        ap_scores.append(ap / len(relevant))
        for k in top_k_list:
            r = len(set(ranked[:k]) & relevant) / len(relevant)
            recalls[k].append(r)
    results = {
        "MAP"      : round(np.mean(ap_scores), 4),
        "Recall@5" : round(np.mean(recalls[5]), 4),
        "Recall@10": round(np.mean(recalls[10]), 4),
    }
    if label:
        print(f"\n=== {label} ===")
        for k, v in results.items():
            print(f"  {k:12}: {v}")
    return results

# ── semantic_retrieve ────────────────────────────────────
def semantic_retrieve(q_emb, corpus_embeddings, top_k=10):
    scores = corpus_embeddings @ q_emb
    ranked = np.argsort(scores)[::-1][:top_k]
    return [(int(idx), float(scores[idx])) for idx in ranked]

# ── embed_long_query ─────────────────────────────────────
def embed_long_query(text, model, tokenizer,
                     chunk_size=400, overlap=50):
    tokens = tokenizer(
        text, truncation=False, return_tensors=None
    )["input_ids"]
    if len(tokens) <= 512:
        prefixed = (
            "Represent this sentence for searching "
            f"relevant passages: {text}"
        )
        return model.encode(
            [prefixed],
            normalize_embeddings=True,
            convert_to_numpy=True
        )[0]
    chunks = []
    start  = 0
    while start < len(tokens):
        end        = min(start + chunk_size, len(tokens))
        chunk_text = tokenizer.decode(
            tokens[start:end], skip_special_tokens=True
        )
        chunks.append(
            "Represent this sentence for searching "
            f"relevant passages: {chunk_text}"
        )
        if end == len(tokens):
            break
        start += chunk_size - overlap
    chunk_embs = model.encode(
        chunks,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )
    mean_emb = chunk_embs.mean(axis=0)
    return mean_emb / np.linalg.norm(mean_emb)

# ── LegalDocumentGraph ───────────────────────────────────
class LegalDocumentGraph:
    def __init__(self, embeddings, metric="mahalanobis"):
        self.embeddings = embeddings
        self.N          = len(embeddings)
        self.metric     = metric
        self.k          = max(3, int(np.ceil(np.log(self.N))))
        print(f"Building {metric} graph | N={self.N} | K={self.k}")
        self.dist_matrix = self._compute_distances()
        self.graph       = self._build_graph()
        print(f"Edges: {self.graph.number_of_edges()}")

    def _compute_distances(self):
        if self.metric == "euclidean":
            return euclidean_distances(self.embeddings)
        elif self.metric == "cosine":
            return cosine_distances(self.embeddings)
        elif self.metric == "mahalanobis":
            print("  Fitting LedoitWolf...")
            lw = LedoitWolf()
            lw.fit(self.embeddings)
            precision     = lw.get_precision()
            L             = np.linalg.cholesky(precision)
            X_transformed = self.embeddings @ L
            return euclidean_distances(X_transformed)

    def _build_graph(self):
        G = nx.Graph()
        G.add_nodes_from(range(self.N))
        for i in range(self.N):
            nearest = np.argsort(self.dist_matrix[i])[1:self.k+1]
            for j in nearest:
                w = 1.0 / (self.dist_matrix[i][j] + 1e-9)
                G.add_edge(i, j, weight=w)
        return G

    def get_neighbors(self, node, max_hops=2):
        return {
            n: d for n, d in
            nx.single_source_shortest_path_length(
                self.graph, node, cutoff=max_hops
            ).items() if n != node
        }

    def get_edge_weight(self, i, j):
        return self.graph[i][j]["weight"] \
               if self.graph.has_edge(i, j) else 0.0

# ── SSHRetriever ─────────────────────────────────────────
class SSHRetriever:
    def __init__(self, corpus_texts, corpus_embeddings,
                 graph, query_model, query_tokenizer,
                 decay_lambda=0.7, alpha=0.5,
                 bm25_top_k=20, sem_top_k=20,
                 max_hops=2, top_k=10):
        self.corpus_texts      = corpus_texts
        self.corpus_embeddings = corpus_embeddings
        self.graph             = graph
        self.query_model       = query_model
        self.query_tokenizer   = query_tokenizer
        self.decay_lambda      = decay_lambda
        self.alpha             = alpha
        self.bm25_top_k        = bm25_top_k
        self.sem_top_k         = sem_top_k
        self.max_hops          = max_hops
        self.top_k             = top_k
        tokenized   = [t.lower().split() for t in corpus_texts]
        self.bm25   = BM25Okapi(tokenized)

    def _bm25_anchors(self, query_text):
        tokens = query_text.lower().split()
        scores = self.bm25.get_scores(tokens)
        ranked = np.argsort(scores)[::-1][:self.bm25_top_k]
        s      = scores[ranked]
        if s.max() > 0:
            s = s / s.max()
        return ranked, s

    def _semantic_anchors(self, q_emb):
        scores = self.corpus_embeddings @ q_emb
        ranked = np.argsort(scores)[::-1][:self.sem_top_k]
        s      = scores[ranked]
        if s.max() > s.min():
            s = (s - s.min()) / (s.max() - s.min())
        return ranked, s

    def _propagate(self, anchor_indices, anchor_scores):
        graph_scores = defaultdict(float)
        for idx, score in zip(anchor_indices, anchor_scores):
            graph_scores[int(idx)] += float(score)
            for neighbor, hops in self.graph.get_neighbors(
                int(idx), self.max_hops
            ).items():
                ew = self.graph.get_edge_weight(int(idx), neighbor)
                graph_scores[neighbor] += (
                    float(score) * ew *
                    (self.decay_lambda ** hops)
                )
        return graph_scores

    def retrieve_from_embedding(self, q_emb, query_text):
        bm25_idx, bm25_sc = self._bm25_anchors(query_text)
        sem_idx,  sem_sc  = self._semantic_anchors(q_emb)
        bm25_graph = self._propagate(bm25_idx, bm25_sc)
        sem_graph  = self._propagate(sem_idx,  sem_sc)
        all_cands  = set(bm25_graph) | set(sem_graph)
        merged     = {
            c: (bm25_graph.get(c,0.0) + sem_graph.get(c,0.0))/2.0
            for c in all_cands
        }
        cand_list = list(merged.keys())
        cand_embs = self.corpus_embeddings[cand_list]
        sem_final = cand_embs @ q_emb
        g = np.array([merged[c] for c in cand_list])
        if g.max() > 0:
            g = g / g.max()
        s = sem_final
        if s.max() > s.min():
            s = (s - s.min()) / (s.max() - s.min())
        final   = self.alpha * s + (1 - self.alpha) * g
        top_idx = np.argsort(final)[::-1][:self.top_k]
        return [(cand_list[i], float(final[i])) for i in top_idx]

print("All functions defined.")

All functions defined.


In [21]:
import random
from rank_bm25 import BM25Okapi 
random.seed(42)

# Split — 160 dev, 40 test
all_qids   = list(gold_labels.keys())
random.shuffle(all_qids)
dev_ids    = all_qids[:160]
test_ids   = all_qids[160:]

dev_q      = {q: q for q in dev_ids}
test_q     = {q: q for q in test_ids}
dev_r      = {q: {gold_labels[q]} for q in dev_ids}
test_r     = {q: {gold_labels[q]} for q in test_ids}

print(f"Dev : {len(dev_ids)} questions")
print(f"Test: {len(test_ids)} questions")
print(f"Corpus: {len(chunk_texts)} chunks")

# Embed all questions
print("\nEmbedding 200 questions...")
question_embeddings = {}
for q in all_qids:
    question_embeddings[q] = model.encode(
        [f"Represent this sentence for searching relevant passages: {q}"],
        normalize_embeddings = True,
        convert_to_numpy     = True
    )[0]
print("Done.")

# BM25 baseline
print("\nRunning BM25 baseline...")
from rank_bm25 import BM25Okapi
tokenized_chunks = [t.lower().split() for t in chunk_texts]
bm25_duomind     = BM25Okapi(tokenized_chunks)

bm25_retrieved = {}
for q in all_qids:
    tokens = q.lower().split()
    scores = bm25_duomind.get_scores(tokens)
    ranked = np.argsort(scores)[::-1][:10]
    bm25_retrieved[q] = [(int(i), float(scores[i])) for i in ranked]

r_bm25_dm = evaluate(
    bm25_retrieved, test_r, test_ids,
    label="DuoMind BM25"
)

# BGE baseline
print("\nRunning BGE baseline...")
bge_retrieved = {
    q: semantic_retrieve(question_embeddings[q], duomind_embeddings)
    for q in all_qids
}
r_bge_dm = evaluate(
    bge_retrieved, test_r, test_ids,
    label="DuoMind BGE Chunked"
)

# Build graphs
print("\nBuilding graphs...")
from sklearn.metrics.pairwise import euclidean_distances, cosine_distances
from sklearn.covariance import LedoitWolf
import networkx as nx

graph_euclid_dm = LegalDocumentGraph(duomind_embeddings, "euclidean")
graph_cosine_dm = LegalDocumentGraph(duomind_embeddings, "cosine")
graph_maha_dm   = LegalDocumentGraph(duomind_embeddings, "mahalanobis")
print("All graphs built.")

# Quick param validation on dev
print("\nValidating params (Mahalanobis)...")
lambda_values = [0.5, 0.6, 0.7, 0.8, 0.9]
alpha_values  = [0.3, 0.4, 0.5, 0.6, 0.7]
best_map      = 0
best_params_dm= {}

for lam in lambda_values:
    for alpha in alpha_values:
        retriever = SSHRetriever(
            corpus_texts      = chunk_texts,
            corpus_embeddings = duomind_embeddings,
            graph             = graph_maha_dm,
            query_model       = model,
            query_tokenizer   = tokenizer,
            decay_lambda      = lam,
            alpha             = alpha
        )
        retrieved = {
            q: retriever.retrieve_from_embedding(
                question_embeddings[q], q
            )
            for q in dev_ids
        }
        results = evaluate(
            retrieved, dev_r, dev_ids, label=""
        )
        if results["MAP"] > best_map:
            best_map      = results["MAP"]
            best_params_dm= {"lambda": lam, "alpha": alpha}

print(f"Best params: {best_params_dm} | MAP={best_map:.4f}")

# Final test evaluation — all three metrics
print("\nRunning final test evaluation...")
results_dm = {}

for metric, graph in [
    ("Euclidean",   graph_euclid_dm),
    ("Cosine",      graph_cosine_dm),
    ("Mahalanobis", graph_maha_dm),
]:
    retriever = SSHRetriever(
        corpus_texts      = chunk_texts,
        corpus_embeddings = duomind_embeddings,
        graph             = graph,
        query_model       = model,
        query_tokenizer   = tokenizer,
        decay_lambda      = best_params_dm["lambda"],
        alpha             = best_params_dm["alpha"]
    )
    retrieved = {
        q: retriever.retrieve_from_embedding(
            question_embeddings[q], q
        )
        for q in test_ids
    }
    results_dm[f"SSH-RAG ({metric})"] = evaluate(
        retrieved, test_r, test_ids,
        label=f"DuoMind SSH-RAG {metric}"
    )

# Final table
print("\n" + "="*60)
print("DUOMIND FINAL RESULTS — TEST SET (40 questions)")
print("="*60)
print(f"{'Method':<30} {'MAP':>8} {'R@5':>8} {'R@10':>8}")
print("-"*60)
for label, res in [
    ("BM25",        r_bm25_dm),
    ("BGE Chunked", r_bge_dm),
    *[(k,v) for k,v in results_dm.items()]
]:
    print(f"{label:<30} "
          f"{res['MAP']:>8.4f} "
          f"{res['Recall@5']:>8.4f} "
          f"{res['Recall@10']:>8.4f}")
print("="*60)

Dev : 160 questions
Test: 40 questions
Corpus: 901 chunks

Embedding 200 questions...
Done.

Running BM25 baseline...

=== DuoMind BM25 ===
  MAP         : 0.0836
  Recall@5    : 0.15
  Recall@10   : 0.3

Running BGE baseline...

=== DuoMind BGE Chunked ===
  MAP         : 0.2509
  Recall@5    : 0.45
  Recall@10   : 0.55

Building graphs...
Building euclidean graph | N=901 | K=7
Edges: 4442
Building cosine graph | N=901 | K=7
Edges: 4442
Building mahalanobis graph | N=901 | K=7
  Fitting LedoitWolf...
Edges: 5892
All graphs built.

Validating params (Mahalanobis)...
Best params: {'lambda': 0.7, 'alpha': 0.5} | MAP=0.2996

Running final test evaluation...

=== DuoMind SSH-RAG Euclidean ===
  MAP         : 0.243
  Recall@5    : 0.325
  Recall@10   : 0.425

=== DuoMind SSH-RAG Cosine ===
  MAP         : 0.1691
  Recall@5    : 0.3
  Recall@10   : 0.4

=== DuoMind SSH-RAG Mahalanobis ===
  MAP         : 0.2421
  Recall@5    : 0.425
  Recall@10   : 0.5

DUOMIND FINAL RESULTS — TEST SET (40 q

#generating triplets

In [1]:
!pip install rank_bm25 sentence-transformers scikit-learn networkx -q

import os
import re
import json
import random
import numpy as np
import networkx as nx
import pandas as pd
from pathlib import Path
from collections import defaultdict
from rank_bm25 import BM25Okapi
from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses
)
from sklearn.covariance import LedoitWolf
from sklearn.metrics.pairwise import euclidean_distances, cosine_distances
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"GPU   : {torch.cuda.get_device_name(0)}")

Device: cuda
GPU   : Tesla T4


/tmp/ipykernel_58/2263882691.py:13: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (


In [3]:
def evaluate(retrieved_dict, relevance, query_ids,
             top_k_list=[5,10], label=""):
    ap_scores = []
    recalls   = {k: [] for k in top_k_list}
    for q_id in query_ids:
        if q_id not in relevance or not relevance[q_id]:
            continue
        relevant = relevance[q_id]
        ranked   = [r[0] for r in retrieved_dict[q_id]]
        hits, ap = 0, 0.0
        for rank, idx in enumerate(ranked, 1):
            if idx in relevant:
                hits += 1
                ap   += hits / rank
        ap_scores.append(ap / len(relevant))
        for k in top_k_list:
            r = len(set(ranked[:k]) & relevant) / len(relevant)
            recalls[k].append(r)
    results = {
        "MAP"      : round(np.mean(ap_scores), 4),
        "Recall@5" : round(np.mean(recalls[5]), 4),
        "Recall@10": round(np.mean(recalls[10]), 4),
    }
    if label:
        print(f"\n=== {label} ===")
        for k, v in results.items():
            print(f"  {k:12}: {v}")
    return results

def semantic_retrieve(q_emb, corpus_embeddings, top_k=10):
    scores = corpus_embeddings @ q_emb
    ranked = np.argsort(scores)[::-1][:top_k]
    return [(int(idx), float(scores[idx])) for idx in ranked]

def embed_long_query(text, model, tokenizer,
                     chunk_size=400, overlap=50):
    tokens = tokenizer(
        text, truncation=False, return_tensors=None
    )["input_ids"]
    if len(tokens) <= 512:
        prefixed = (
            "Represent this sentence for searching "
            f"relevant passages: {text}"
        )
        return model.encode(
            [prefixed],
            normalize_embeddings=True,
            convert_to_numpy=True
        )[0]
    chunks = []
    start  = 0
    while start < len(tokens):
        end        = min(start + chunk_size, len(tokens))
        chunk_text = tokenizer.decode(
            tokens[start:end], skip_special_tokens=True
        )
        chunks.append(
            "Represent this sentence for searching "
            f"relevant passages: {chunk_text}"
        )
        if end == len(tokens):
            break
        start += chunk_size - overlap
    chunk_embs = model.encode(
        chunks,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )
    mean_emb = chunk_embs.mean(axis=0)
    return mean_emb / np.linalg.norm(mean_emb)

class LegalDocumentGraph:
    def __init__(self, embeddings, metric="mahalanobis"):
        self.embeddings = embeddings
        self.N          = len(embeddings)
        self.metric     = metric
        self.k          = max(3, int(np.ceil(np.log(self.N))))
        print(f"Building {metric} graph | N={self.N} | K={self.k}")
        self.dist_matrix = self._compute_distances()
        self.graph       = self._build_graph()
        print(f"Edges: {self.graph.number_of_edges()}")

    def _compute_distances(self):
        if self.metric == "euclidean":
            return euclidean_distances(self.embeddings)
        elif self.metric == "cosine":
            return cosine_distances(self.embeddings)
        elif self.metric == "mahalanobis":
            print("  Fitting LedoitWolf...")
            lw = LedoitWolf()
            lw.fit(self.embeddings)
            precision     = lw.get_precision()
            L             = np.linalg.cholesky(precision)
            X_transformed = self.embeddings @ L
            return euclidean_distances(X_transformed)

    def _build_graph(self):
        G = nx.Graph()
        G.add_nodes_from(range(self.N))
        for i in range(self.N):
            nearest = np.argsort(self.dist_matrix[i])[1:self.k+1]
            for j in nearest:
                w = 1.0 / (self.dist_matrix[i][j] + 1e-9)
                G.add_edge(i, j, weight=w)
        return G

    def get_neighbors(self, node, max_hops=2):
        return {
            n: d for n, d in
            nx.single_source_shortest_path_length(
                self.graph, node, cutoff=max_hops
            ).items() if n != node
        }

    def get_edge_weight(self, i, j):
        return self.graph[i][j]["weight"] \
               if self.graph.has_edge(i, j) else 0.0

class SSHRetriever:
    def __init__(self, corpus_texts, corpus_embeddings,
                 graph, query_model, query_tokenizer,
                 decay_lambda=0.7, alpha=0.5,
                 bm25_top_k=20, sem_top_k=20,
                 max_hops=2, top_k=10):
        self.corpus_texts      = corpus_texts
        self.corpus_embeddings = corpus_embeddings
        self.graph             = graph
        self.query_model       = query_model
        self.query_tokenizer   = query_tokenizer
        self.decay_lambda      = decay_lambda
        self.alpha             = alpha
        self.bm25_top_k        = bm25_top_k
        self.sem_top_k         = sem_top_k
        self.max_hops          = max_hops
        self.top_k             = top_k
        tokenized   = [t.lower().split() for t in corpus_texts]
        self.bm25   = BM25Okapi(tokenized)

    def _bm25_anchors(self, query_text):
        tokens = query_text.lower().split()
        scores = self.bm25.get_scores(tokens)
        ranked = np.argsort(scores)[::-1][:self.bm25_top_k]
        s      = scores[ranked]
        if s.max() > 0:
            s = s / s.max()
        return ranked, s

    def _semantic_anchors(self, q_emb):
        scores = self.corpus_embeddings @ q_emb
        ranked = np.argsort(scores)[::-1][:self.sem_top_k]
        s      = scores[ranked]
        if s.max() > s.min():
            s = (s - s.min()) / (s.max() - s.min())
        return ranked, s

    def _propagate(self, anchor_indices, anchor_scores):
        graph_scores = defaultdict(float)
        for idx, score in zip(anchor_indices, anchor_scores):
            graph_scores[int(idx)] += float(score)
            for neighbor, hops in self.graph.get_neighbors(
                int(idx), self.max_hops
            ).items():
                ew = self.graph.get_edge_weight(int(idx), neighbor)
                graph_scores[neighbor] += (
                    float(score) * ew *
                    (self.decay_lambda ** hops)
                )
        return graph_scores

    def retrieve_from_embedding(self, q_emb, query_text):
        bm25_idx, bm25_sc = self._bm25_anchors(query_text)
        sem_idx,  sem_sc  = self._semantic_anchors(q_emb)
        bm25_graph = self._propagate(bm25_idx, bm25_sc)
        sem_graph  = self._propagate(sem_idx,  sem_sc)
        all_cands  = set(bm25_graph) | set(sem_graph)
        merged     = {
            c: (bm25_graph.get(c,0.0) + sem_graph.get(c,0.0))/2.0
            for c in all_cands
        }
        cand_list = list(merged.keys())
        cand_embs = self.corpus_embeddings[cand_list]
        sem_final = cand_embs @ q_emb
        g = np.array([merged[c] for c in cand_list])
        if g.max() > 0:
            g = g / g.max()
        s = sem_final
        if s.max() > s.min():
            s = (s - s.min()) / (s.max() - s.min())
        final   = self.alpha * s + (1 - self.alpha) * g
        top_idx = np.argsort(final)[::-1][:self.top_k]
        return [(cand_list[i], float(final[i])) for i in top_idx]

print("All functions defined.")

All functions defined.


In [13]:
print("Loading BGE...")
model     = SentenceTransformer("BAAI/bge-base-en-v1.5", device="cuda")
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")
print("Done.")

Loading BGE...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Done.


In [20]:
!pip install pymupdf python-docx -q
import fitz
from docx import Document as DocxDocument

def load_document(filepath):
    ext = Path(filepath).suffix.lower()
    if ext == ".pdf":
        doc  = fitz.open(filepath)
        text = ""
        for page in doc:
            text += page.get_text()
        doc.close()
        return text
    elif ext == ".docx":
        doc   = DocxDocument(filepath)
        paras = [p.text for p in doc.paragraphs if p.text.strip()]
        return "\n".join(paras)
    elif ext == ".txt":
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    return ""

def chunk_text(text, doc_name, chunk_size=256, overlap=50):
    words  = text.split()
    chunks = []
    start  = 0
    idx    = 0
    while start < len(words):
        end  = min(start + chunk_size, len(words))
        chunks.append({
            "chunk_id": f"{doc_name}_chunk_{idx}",
            "doc_name": doc_name,
            "text"    : " ".join(words[start:end])
        })
        if end == len(words):
            break
        start += chunk_size - overlap
        idx   += 1
    return chunks

def load_all_documents(doc_dir):
    all_chunks = []
    doc_files  = [
        f for f in os.listdir(doc_dir)
        if f.endswith((".pdf", ".docx", ".txt"))
    ]
    print(f"Found {len(doc_files)} documents")
    for fname in doc_files:
        fpath    = os.path.join(doc_dir, fname)
        doc_name = Path(fname).stem
        text     = load_document(fpath)
        if not text.strip():
            continue
        chunks = chunk_text(text, doc_name)
        all_chunks.extend(chunks)
        print(f"  {fname}: {len(chunks)} chunks")
    print(f"Total: {len(all_chunks)} chunks")
    return all_chunks

doc_dir    = "/kaggle/input/datasets/sauraviitjapdcl/duomind-sshrag-mahalanobis/"  # update path
all_chunks = load_all_documents(doc_dir)
chunk_texts= [c["text"] for c in all_chunks]
chunk_ids  = [c["chunk_id"] for c in all_chunks]

Found 20 documents
  24062022_Guidelines_on_Cyber_Security_in_Power_Sector_2021.pdf: 40 chunks
  NCIIPC_Guidelines_V2.pdf: 72 chunks
  CERT-In_Directions.docx: 9 chunks
  DPDP ACT.pdf: 52 chunks
  State Data Centre security guidelines.pdf: 13 chunks
  DPDP_Rules_2025_English_only.pdf: 50 chunks
  CELEX_32016R0679_EN_TXT.pdf: 271 chunks
  Presentation - CSCRF.pdf: 10 chunks
  e-Governance Policy Initiatives.pdf: 213 chunks
  Electricity Act 2003.pdf: 200 chunks
  guidelinesgovtentities.pdf: 74 chunks
  Cyber Security Framework in Banks.docx: 33 chunks
  ISO_IEC-270012022-ed.3.pdf: 35 chunks
  nist cyber security framework.docx: 38 chunks
  trai_takshalia.pdf: 20 chunks
  NIST.SP.800-66r2.pdf: 205 chunks
  OECD-LEGAL-0479-en.docx: 20 chunks
  pcissc_overview.pdf: 6 chunks
  NCIIPC-Rules-notification-1.pdf: 20 chunks
  IT ACT 2000.pdf: 95 chunks
Total: 1476 chunks


In [11]:
# Show cross-document questions
cross = qa_df_clean[
    qa_df_clean['document_source'].str.contains('\+|Cross')
]

print("Sample cross-document questions:")
print(cross[['question', 'ground_truth_answer', 'document_source']].head(10).to_string())

Sample cross-document questions:
                                                                                        question                                                                                                                                   ground_truth_answer document_source
110           How do the ISMS antivirus policy and remote working policy interact for VPN users?                                 VPN encrypts traffic, but endpoint antivirus is still required as VPN does not protect against malware on the device.  Cross-Document
111                    How do DPDP Act obligations interact with the ISMS access control policy?                     Access control limits who can view personal data, directly supporting DPDP's purpose limitation and data minimization principles.  Cross-Document
112            How does the CCMP support compliance with NCIIPC incident reporting requirements?                                              The CCMP's external notification pro

<>:3: SyntaxWarning: invalid escape sequence '\+'
<>:3: SyntaxWarning: invalid escape sequence '\+'
/tmp/ipykernel_58/2712294911.py:3: SyntaxWarning: invalid escape sequence '\+'
  qa_df_clean['document_source'].str.contains('\+|Cross')


In [23]:
# Reload 340Q with encoding fix
qa_df = pd.read_csv(
    "/kaggle/input/datasets/sauraviitjapdcl/ssh-rag-code/ssh_rag_groundtruth.csv",
    encoding="cp1252"
)
qa_df_clean = qa_df.drop_duplicates(subset='question', keep='first')
print(f"Unique questions: {len(qa_df_clean)}")
print(f"Corpus chunks   : {len(chunk_texts)}")

# Embed chunks
print("\nEmbedding 1476 chunks...")
duomind_embeddings = model.encode(
    chunk_texts,
    batch_size           = 64,
    show_progress_bar    = True,
    normalize_embeddings = True,
    convert_to_numpy     = True
)
print(f"Shape: {duomind_embeddings.shape}")

# Embed answers
questions_list = qa_df_clean["question"].tolist()
answer_texts   = qa_df_clean["ground_truth_answer"].tolist()

print("\nEmbedding answers...")
answer_embeddings = model.encode(
    answer_texts,
    batch_size           = 64,
    show_progress_bar    = True,
    normalize_embeddings = True,
    convert_to_numpy     = True
)

# Build gold labels with dual support for cross-document
print("\nBuilding gold labels...")
gold_labels = {}
gold_scores = []

for i in range(len(qa_df_clean)):
    question   = qa_df_clean.iloc[i]["question"]
    source     = str(qa_df_clean.iloc[i]["document_source"])
    answer_emb = answer_embeddings[i]

    scores   = duomind_embeddings @ answer_emb
    is_cross = '+' in source or 'Cross' in source

    if is_cross:
        top2_idx = np.argsort(scores)[::-1][:2]
        gold_labels[question] = set(top2_idx.tolist())
    else:
        best_idx = int(np.argmax(scores))
        gold_labels[question] = {best_idx}

    gold_scores.append(float(np.max(scores)))

print(f"\nGold labels built : {len(gold_labels)}")
print(f"Avg score         : {np.mean(gold_scores):.4f}")
print(f"High conf (>0.75) : {sum(1 for s in gold_scores if s > 0.75)}")
print(f"Medium (0.6-0.75) : {sum(1 for s in gold_scores if 0.6 <= s <= 0.75)}")
print(f"Low (<0.6)        : {sum(1 for s in gold_scores if s < 0.6)}")
print(f"Single-doc labels : {sum(1 for v in gold_labels.values() if len(v)==1)}")
print(f"Cross-doc labels  : {sum(1 for v in gold_labels.values() if len(v)==2)}")

Unique questions: 378
Corpus chunks   : 1476

Embedding 1476 chunks...


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Shape: (1476, 768)

Embedding answers...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]


Building gold labels...

Gold labels built : 378
Avg score         : 0.7138
High conf (>0.75) : 98
Medium (0.6-0.75) : 273
Low (<0.6)        : 7
Single-doc labels : 288
Cross-doc labels  : 90


In [22]:
from collections import Counter

doc_names = [c["doc_name"] for c in all_chunks]
counts    = Counter(doc_names)

print("=== LOADED DOCUMENTS ===")
for doc, count in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {count:4d} chunks — {doc}")
print(f"\nTotal: {len(all_chunks)} chunks")

=== LOADED DOCUMENTS ===
   271 chunks — CELEX_32016R0679_EN_TXT
   213 chunks — e-Governance Policy Initiatives
   205 chunks — NIST.SP.800-66r2
   200 chunks — Electricity Act 2003
    95 chunks — IT ACT 2000
    74 chunks — guidelinesgovtentities
    72 chunks — NCIIPC_Guidelines_V2
    52 chunks — DPDP ACT
    50 chunks — DPDP_Rules_2025_English_only
    40 chunks — 24062022_Guidelines_on_Cyber_Security_in_Power_Sector_2021
    38 chunks — nist cyber security framework
    35 chunks — ISO_IEC-270012022-ed.3
    33 chunks — Cyber Security Framework in Banks
    20 chunks — trai_takshalia
    20 chunks — OECD-LEGAL-0479-en
    20 chunks — NCIIPC-Rules-notification-1
    13 chunks — State Data Centre security guidelines
    10 chunks — Presentation - CSCRF
     9 chunks — CERT-In_Directions
     6 chunks — pcissc_overview

Total: 1476 chunks


In [21]:
# Check which documents already have questions
print(qa_df['document_source'].value_counts())

document_source
DPDP Act 2023                                                                                  40
NCIIPC Guidelines                                                                              40
ISO 27001:2022                                                                                 30
Cross-Document                                                                                 30
Presentation - CSCRF(1).pdf                                                                    27
e-Governance Policy India + CERT-In Directions 2022                                            26
State Data Centre security guidelines                                                          26
GDPR                                                                                           26
PCI Security Standards – Standards Overview                                                    25
Digital Personal Data Protection Rules, 2025                                                   25
Priv

In [24]:
import random
random.seed(42)

# Split 80/20 dev/test
all_qids  = list(gold_labels.keys())
random.shuffle(all_qids)
dev_ids   = all_qids[:302]   # 80%
test_ids  = all_qids[302:]   # 20% = 76 questions

dev_r  = {q: gold_labels[q] for q in dev_ids}
test_r = {q: gold_labels[q] for q in test_ids}

print(f"Dev : {len(dev_ids)} questions")
print(f"Test: {len(test_ids)} questions")
print(f"Corpus: {len(chunk_texts)} chunks")

# Embed all questions
print("\nEmbedding 378 questions...")
question_embeddings = {}
for q in all_qids:
    question_embeddings[q] = model.encode(
        [f"Represent this sentence for searching relevant passages: {q}"],
        normalize_embeddings = True,
        convert_to_numpy     = True
    )[0]
print("Done.")

# Updated evaluate function — handles set-based gold labels
def evaluate_dual(retrieved_dict, relevance, query_ids,
                  top_k_list=[5,10], label=""):
    ap_scores = []
    recalls   = {k: [] for k in top_k_list}
    for q_id in query_ids:
        if q_id not in relevance or not relevance[q_id]:
            continue
        relevant = relevance[q_id]
        ranked   = [r[0] for r in retrieved_dict[q_id]]
        hits, ap = 0, 0.0
        for rank, idx in enumerate(ranked, 1):
            if idx in relevant:
                hits += 1
                ap   += hits / rank
        ap_scores.append(ap / len(relevant))
        for k in top_k_list:
            r = len(set(ranked[:k]) & relevant) / len(relevant)
            recalls[k].append(r)
    results = {
        "MAP"      : round(np.mean(ap_scores), 4),
        "Recall@5" : round(np.mean(recalls[5]), 4),
        "Recall@10": round(np.mean(recalls[10]), 4),
    }
    if label:
        print(f"\n=== {label} ===")
        for k, v in results.items():
            print(f"  {k:12}: {v}")
    return results

# BM25 baseline
print("\nRunning BM25...")
tokenized_chunks = [t.lower().split() for t in chunk_texts]
bm25_dm          = BM25Okapi(tokenized_chunks)

bm25_ret = {}
for q in test_ids:
    tokens      = q.lower().split()
    scores      = bm25_dm.get_scores(tokens)
    ranked      = np.argsort(scores)[::-1][:10]
    bm25_ret[q] = [(int(i), float(scores[i])) for i in ranked]

r_bm25 = evaluate_dual(
    bm25_ret, test_r, test_ids,
    label="DuoMind BM25"
)

# BGE baseline
print("\nRunning BGE...")
bge_ret = {
    q: semantic_retrieve(question_embeddings[q], duomind_embeddings)
    for q in test_ids
}
r_bge = evaluate_dual(
    bge_ret, test_r, test_ids,
    label="DuoMind BGE"
)

# Build graphs
print("\nBuilding graphs...")
graph_euclid_dm = LegalDocumentGraph(duomind_embeddings, "euclidean")
graph_cosine_dm = LegalDocumentGraph(duomind_embeddings, "cosine")
graph_maha_dm   = LegalDocumentGraph(duomind_embeddings, "mahalanobis")
print("All graphs built.")

# Validate params on dev — Mahalanobis only
print("\nValidating params...")
best_map_dm    = 0
best_params_dm = {}

dev_sample = random.sample(dev_ids, min(150, len(dev_ids)))

for lam in [0.5, 0.6, 0.7, 0.8, 0.9]:
    for alpha in [0.3, 0.4, 0.5, 0.6, 0.7]:
        ret = SSHRetriever(
            corpus_texts      = chunk_texts,
            corpus_embeddings = duomind_embeddings,
            graph             = graph_maha_dm,
            query_model       = model,
            query_tokenizer   = tokenizer,
            decay_lambda      = lam,
            alpha             = alpha
        )
        retrieved = {
            q: ret.retrieve_from_embedding(
                question_embeddings[q], q
            )
            for q in dev_sample
        }
        res = evaluate_dual(retrieved, dev_r, dev_sample, label="")
        if res["MAP"] > best_map_dm:
            best_map_dm    = res["MAP"]
            best_params_dm = {"lambda": lam, "alpha": alpha}

print(f"Best params: {best_params_dm} | MAP={best_map_dm:.4f}")

# Final test — all three variants
results_dm = {}
for metric, graph in [
    ("Euclidean",   graph_euclid_dm),
    ("Cosine",      graph_cosine_dm),
    ("Mahalanobis", graph_maha_dm),
]:
    ret = SSHRetriever(
        corpus_texts      = chunk_texts,
        corpus_embeddings = duomind_embeddings,
        graph             = graph,
        query_model       = model,
        query_tokenizer   = tokenizer,
        decay_lambda      = best_params_dm["lambda"],
        alpha             = best_params_dm["alpha"]
    )
    retrieved = {
        q: ret.retrieve_from_embedding(
            question_embeddings[q], q
        )
        for q in test_ids
    }
    results_dm[f"SSH-RAG ({metric})"] = evaluate_dual(
        retrieved, test_r, test_ids,
        label=f"DuoMind SSH-RAG {metric}"
    )

# Final table
print("\n" + "="*65)
print("DUOMIND FINAL — TEST SET (76 questions, 1476 chunks)")
print("="*65)
print(f"{'Method':<30} {'MAP':>8} {'R@5':>8} {'R@10':>8}")
print("-"*65)
for label, res in [
    ("BM25",        r_bm25),
    ("BGE Chunked", r_bge),
    *[(k,v) for k,v in results_dm.items()]
]:
    print(f"{label:<30} "
          f"{res['MAP']:>8.4f} "
          f"{res['Recall@5']:>8.4f} "
          f"{res['Recall@10']:>8.4f}")
print("="*65)

Dev : 302 questions
Test: 76 questions
Corpus: 1476 chunks

Embedding 378 questions...
Done.

Running BM25...

=== DuoMind BM25 ===
  MAP         : 0.1812
  Recall@5    : 0.2895
  Recall@10   : 0.3487

Running BGE...

=== DuoMind BGE ===
  MAP         : 0.3452
  Recall@5    : 0.4539
  Recall@10   : 0.5526

Building graphs...
Building euclidean graph | N=1476 | K=8
Edges: 8309
Building cosine graph | N=1476 | K=8
Edges: 8309
Building mahalanobis graph | N=1476 | K=8
  Fitting LedoitWolf...
Edges: 10853
All graphs built.

Validating params...
Best params: {'lambda': 0.6, 'alpha': 0.7} | MAP=0.3995

=== DuoMind SSH-RAG Euclidean ===
  MAP         : 0.3503
  Recall@5    : 0.4276
  Recall@10   : 0.4934

=== DuoMind SSH-RAG Cosine ===
  MAP         : 0.3145
  Recall@5    : 0.4276
  Recall@10   : 0.4671

=== DuoMind SSH-RAG Mahalanobis ===
  MAP         : 0.3654
  Recall@5    : 0.4671
  Recall@10   : 0.5461

DUOMIND FINAL — TEST SET (76 questions, 1476 chunks)
Method                          

##training bge

In [25]:
!pip install rank_bm25 sentence-transformers scikit-learn networkx pymupdf python-docx -q
!git clone https://github.com/wasiahmad/PolicyQA /kaggle/working/PolicyQA 2>/dev/null || echo "Already exists"

In [26]:
import os
import re
import json
import random
import numpy as np
import networkx as nx
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
from rank_bm25 import BM25Okapi
from sentence_transformers import (
    SentenceTransformer, InputExample, losses
)
from sklearn.covariance import LedoitWolf
from sklearn.metrics.pairwise import euclidean_distances, cosine_distances
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch
import fitz
from docx import Document as DocxDocument

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"GPU   : {torch.cuda.get_device_name(0)}")
random.seed(42)

Device: cuda
GPU   : Tesla T4


In [41]:
# ── Evaluation ───────────────────────────────────────────
def evaluate(retrieved_dict, relevance, query_ids,
             top_k_list=[5,10], label=""):
    ap_scores = []
    recalls   = {k: [] for k in top_k_list}
    for q_id in query_ids:
        if q_id not in relevance or not relevance[q_id]:
            continue
        relevant = relevance[q_id]
        ranked   = [r[0] for r in retrieved_dict[q_id]]
        hits, ap = 0, 0.0
        for rank, idx in enumerate(ranked, 1):
            if idx in relevant:
                hits += 1
                ap   += hits / rank
        ap_scores.append(ap / len(relevant))
        for k in top_k_list:
            r = len(set(ranked[:k]) & relevant) / len(relevant)
            recalls[k].append(r)
    results = {
        "MAP"      : round(np.mean(ap_scores), 4),
        "Recall@5" : round(np.mean(recalls[5]), 4),
        "Recall@10": round(np.mean(recalls[10]), 4),
    }
    if label:
        print(f"\n=== {label} ===")
        for k, v in results.items():
            print(f"  {k:12}: {v}")
    return results

# ── Retrieval ─────────────────────────────────────────────
def semantic_retrieve(q_emb, corpus_embeddings, top_k=10):
    scores = corpus_embeddings @ q_emb
    ranked = np.argsort(scores)[::-1][:top_k]
    return [(int(idx), float(scores[idx])) for idx in ranked]

def embed_query(text, model, tokenizer,
                chunk_size=400, overlap=50):
    tokens = tokenizer(
        text, truncation=False, return_tensors=None
    )["input_ids"]
    if len(tokens) <= 512:
        return model.encode(
            [f"Represent this sentence for searching relevant passages: {text}"],
            normalize_embeddings=True,
            convert_to_numpy=True
        )[0]
    chunks = []
    start  = 0
    while start < len(tokens):
        end   = min(start + chunk_size, len(tokens))
        chunk = tokenizer.decode(tokens[start:end],
                                 skip_special_tokens=True)
        chunks.append(
            f"Represent this sentence for searching relevant passages: {chunk}"
        )
        if end == len(tokens):
            break
        start += chunk_size - overlap
    embs     = model.encode(chunks, normalize_embeddings=True,
                            convert_to_numpy=True)
    mean_emb = embs.mean(axis=0)
    return mean_emb / np.linalg.norm(mean_emb)

# ── Graph ─────────────────────────────────────────────────
class LegalDocumentGraph:
    def __init__(self, embeddings, metric="mahalanobis"):
        self.embeddings = embeddings
        self.N          = len(embeddings)
        self.metric     = metric
        self.k          = max(3, int(np.ceil(np.log(self.N))))
        print(f"Building {metric} graph | N={self.N} | K={self.k}")
        self.dist_matrix = self._compute_distances()
        self.graph       = self._build_graph()
        print(f"Edges: {self.graph.number_of_edges()}")

    def _compute_distances(self):
        if self.metric == "euclidean":
            return euclidean_distances(self.embeddings)
        elif self.metric == "cosine":
            return cosine_distances(self.embeddings)
        elif self.metric == "mahalanobis":
            print("  Fitting LedoitWolf...")
            lw = LedoitWolf()
            lw.fit(self.embeddings)
            L  = np.linalg.cholesky(lw.get_precision())
            return euclidean_distances(self.embeddings @ L)

    def _build_graph(self):
        G = nx.Graph()
        G.add_nodes_from(range(self.N))
        for i in range(self.N):
            nearest = np.argsort(self.dist_matrix[i])[1:self.k+1]
            for j in nearest:
                w = 1.0 / (self.dist_matrix[i][j] + 1e-9)
                G.add_edge(i, j, weight=w)
        return G

    def get_neighbors(self, node, max_hops=2):
        return {
            n: d for n, d in
            nx.single_source_shortest_path_length(
                self.graph, node, cutoff=max_hops
            ).items() if n != node
        }

    def get_edge_weight(self, i, j):
        return self.graph[i][j]["weight"] \
               if self.graph.has_edge(i, j) else 0.0

# ── SSH-RAG ───────────────────────────────────────────────
class SSHRetriever:
    def __init__(self, corpus_texts, corpus_embeddings,
                 graph, query_model, query_tokenizer,
                 decay_lambda=0.7, alpha=0.5,
                 bm25_top_k=20, sem_top_k=20,
                 max_hops=2, top_k=10):
        self.corpus_texts      = corpus_texts
        self.corpus_embeddings = corpus_embeddings
        self.graph             = graph
        self.query_model       = query_model
        self.query_tokenizer   = query_tokenizer
        self.decay_lambda      = decay_lambda
        self.alpha             = alpha
        self.bm25_top_k        = bm25_top_k
        self.sem_top_k         = sem_top_k
        self.max_hops          = max_hops
        self.top_k             = top_k
        self.bm25 = BM25Okapi([t.lower().split() for t in corpus_texts])

    def _bm25_anchors(self, query_text):
        scores = self.bm25.get_scores(query_text.lower().split())
        ranked = np.argsort(scores)[::-1][:self.bm25_top_k]
        s      = scores[ranked]
        if s.max() > 0: s = s / s.max()
        return ranked, s

    def _semantic_anchors(self, q_emb):
        scores = self.corpus_embeddings @ q_emb
        ranked = np.argsort(scores)[::-1][:self.sem_top_k]
        s      = scores[ranked]
        if s.max() > s.min():
            s = (s - s.min()) / (s.max() - s.min())
        return ranked, s

    def _propagate(self, anchor_indices, anchor_scores):
        graph_scores = defaultdict(float)
        for idx, score in zip(anchor_indices, anchor_scores):
            graph_scores[int(idx)] += float(score)
            for neighbor, hops in self.graph.get_neighbors(
                int(idx), self.max_hops
            ).items():
                ew = self.graph.get_edge_weight(int(idx), neighbor)
                graph_scores[neighbor] += (
                    float(score) * ew * (self.decay_lambda ** hops)
                )
        return graph_scores

    def retrieve(self, q_emb, query_text):
        bm25_idx, bm25_sc = self._bm25_anchors(query_text)
        sem_idx,  sem_sc  = self._semantic_anchors(q_emb)
        bm25_g = self._propagate(bm25_idx, bm25_sc)
        sem_g  = self._propagate(sem_idx,  sem_sc)
        merged = {
            c: (bm25_g.get(c,0.0) + sem_g.get(c,0.0)) / 2.0
            for c in set(bm25_g) | set(sem_g)
        }
        cand_list = list(merged.keys())
        cand_embs = self.corpus_embeddings[cand_list]
        sem_final = cand_embs @ q_emb
        g = np.array([merged[c] for c in cand_list])
        if g.max() > 0: g = g / g.max()
        s = sem_final
        if s.max() > s.min():
            s = (s - s.min()) / (s.max() - s.min())
        final   = self.alpha * s + (1 - self.alpha) * g
        top_idx = np.argsort(final)[::-1][:self.top_k]
        return [(cand_list[i], float(final[i])) for i in top_idx]

# ── Parameter validation ──────────────────────────────────
# In validate_params — precompute embeddings
def validate_params(corpus_texts, corpus_embeddings,
                    graph, model, tokenizer,
                    dev_ids, dev_q, dev_r,
                    sample=150):
    sample_ids = random.sample(dev_ids, min(sample, len(dev_ids)))
    
    # Embed dev questions first
    print("Embedding dev questions...")
    dev_embs = {}
    for q in sample_ids:
        dev_embs[q] = embed_query(dev_q[q], model, tokenizer)
    
    best_map, best_params = 0, {}

    for lam in [0.5, 0.6, 0.7, 0.8, 0.9]:
        for alpha in [0.3, 0.4, 0.5, 0.6, 0.7]:
            ret = SSHRetriever(
                corpus_texts, corpus_embeddings,
                graph, model, tokenizer,
                decay_lambda=lam, alpha=alpha
            )
            retrieved = {
                q: ret.retrieve(dev_embs[q], dev_q[q])
                for q in sample_ids
            }
            res = evaluate(retrieved, dev_r, sample_ids)
            if res["MAP"] > best_map:
                best_map    = res["MAP"]
                best_params = {"lambda": lam, "alpha": alpha}

    print(f"Best params: {best_params} | MAP={best_map:.4f}")
    return best_params

print("All functions defined.")

All functions defined.


In [28]:
print("Loading BGE...")
model     = SentenceTransformer("BAAI/bge-base-en-v1.5", device="cuda")
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")
print(f"Device: {next(model.parameters()).device}")

Loading BGE...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda:0


In [29]:
def load_policyqa(data_path):
    all_passages  = {}
    passages_list = []
    all_questions = {}
    all_relevance = defaultdict(set)

    for split in ["train.json", "dev.json", "test.json"]:
        with open(f"{data_path}/{split}") as f:
            data = json.load(f)
        for doc in data["data"]:
            title = doc.get("title", "unknown")
            for para in doc["paragraphs"]:
                context = para["context"].strip()
                if context not in all_passages:
                    all_passages[context] = len(passages_list)
                    passages_list.append(context)
                p_idx = all_passages[context]
                for qa in para["qas"]:
                    if qa.get("is_impossible", False): continue
                    if not qa["answers"]: continue
                    q_id   = qa["id"]
                    q_text = f"{title}: {qa['question'].strip()}"
                    all_questions[q_id] = q_text
                    all_relevance[q_id].add(p_idx)

    print(f"PolicyQA passages : {len(passages_list)}")
    print(f"PolicyQA questions: {len(all_questions)}")
    return passages_list, all_questions, dict(all_relevance)

pqa_passages, pqa_questions, pqa_relevance = load_policyqa(
    "/kaggle/working/PolicyQA/data"
)

# Split PolicyQA
all_pqa_ids       = list(pqa_questions.keys())
split_idx         = int(len(all_pqa_ids) * 0.8)
pqa_dev_ids       = all_pqa_ids[:split_idx]
pqa_test_ids      = all_pqa_ids[split_idx:]
pqa_dev_r         = {q: pqa_relevance.get(q, set()) for q in pqa_dev_ids}
pqa_test_r        = {q: pqa_relevance.get(q, set()) for q in pqa_test_ids}
pqa_dev_sample    = random.sample(pqa_dev_ids, 200)

print(f"PolicyQA dev : {len(pqa_dev_ids)}")
print(f"PolicyQA test: {len(pqa_test_ids)}")

PolicyQA passages : 3202
PolicyQA questions: 25017
PolicyQA dev : 20013
PolicyQA test: 5004


In [30]:
def load_document(filepath):
    ext = Path(filepath).suffix.lower()
    if ext == ".pdf":
        doc  = fitz.open(filepath)
        text = "".join(page.get_text() for page in doc)
        doc.close()
        return text
    elif ext == ".docx":
        doc = DocxDocument(filepath)
        return "\n".join(
            p.text for p in doc.paragraphs if p.text.strip()
        )
    elif ext == ".txt":
        for enc in ["utf-8", "latin-1", "cp1252"]:
            try:
                with open(filepath, "r",
                          encoding=enc, errors="ignore") as f:
                    return f.read()
            except: continue
    return ""

def chunk_text(text, doc_name, chunk_size=256, overlap=50):
    words  = text.split()
    chunks = []
    start, idx = 0, 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append({
            "chunk_id": f"{doc_name}_chunk_{idx}",
            "doc_name": doc_name,
            "text"    : " ".join(words[start:end])
        })
        if end == len(words): break
        start += chunk_size - overlap
        idx   += 1
    return chunks

def load_all_documents(doc_dir):
    all_chunks = []
    files = [f for f in os.listdir(doc_dir)
             if f.endswith((".pdf", ".docx", ".txt"))]
    print(f"Found {len(files)} documents")
    for fname in files:
        text   = load_document(os.path.join(doc_dir, fname))
        if not text.strip(): continue
        chunks = chunk_text(text, Path(fname).stem)
        all_chunks.extend(chunks)
        print(f"  {len(chunks):4d} chunks — {fname}")
    print(f"Total: {len(all_chunks)} chunks")
    return all_chunks

doc_dir    = "/kaggle/input/datasets/sauraviitjapdcl/duomind-sshrag-mahalanobis"  # update path
all_chunks = load_all_documents(doc_dir)
chunk_texts= [c["text"] for c in all_chunks]
chunk_ids  = [c["chunk_id"] for c in all_chunks]

Found 20 documents
    40 chunks — 24062022_Guidelines_on_Cyber_Security_in_Power_Sector_2021.pdf
    72 chunks — NCIIPC_Guidelines_V2.pdf
     9 chunks — CERT-In_Directions.docx
    52 chunks — DPDP ACT.pdf
    13 chunks — State Data Centre security guidelines.pdf
    50 chunks — DPDP_Rules_2025_English_only.pdf
   271 chunks — CELEX_32016R0679_EN_TXT.pdf
    10 chunks — Presentation - CSCRF.pdf
   213 chunks — e-Governance Policy Initiatives.pdf
   200 chunks — Electricity Act 2003.pdf
    74 chunks — guidelinesgovtentities.pdf
    33 chunks — Cyber Security Framework in Banks.docx
    35 chunks — ISO_IEC-270012022-ed.3.pdf
    38 chunks — nist cyber security framework.docx
    20 chunks — trai_takshalia.pdf
   205 chunks — NIST.SP.800-66r2.pdf
    20 chunks — OECD-LEGAL-0479-en.docx
     6 chunks — pcissc_overview.pdf
    20 chunks — NCIIPC-Rules-notification-1.pdf
    95 chunks — IT ACT 2000.pdf
Total: 1476 chunks


In [32]:
qa_df = pd.read_csv(
    "/kaggle/input/datasets/sauraviitjapdcl/ssh-rag-code/ssh_rag_groundtruth.csv",
    encoding="cp1252"
)
qa_df_clean = qa_df.drop_duplicates(subset='question', keep='first')
print(f"Unique questions: {len(qa_df_clean)}")

# Embed chunks
print("Embedding chunks...")
duomind_embeddings = model.encode(
    chunk_texts, batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

# Embed answers
answer_texts = qa_df_clean["ground_truth_answer"].tolist()
print("Embedding answers...")
answer_embeddings = model.encode(
    answer_texts, batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

# Build gold labels
gold_labels = {}
gold_scores = []
for i in range(len(qa_df_clean)):
    question   = qa_df_clean.iloc[i]["question"]
    source     = str(qa_df_clean.iloc[i]["document_source"])
    answer_emb = answer_embeddings[i]
    scores     = duomind_embeddings @ answer_emb
    is_cross   = '+' in source or 'Cross' in source
    if is_cross:
        top2 = np.argsort(scores)[::-1][:2]
        gold_labels[question] = set(top2.tolist())
    else:
        gold_labels[question] = {int(np.argmax(scores))}
    gold_scores.append(float(np.max(scores)))

print(f"Gold labels: {len(gold_labels)}")
print(f"Avg score  : {np.mean(gold_scores):.4f}")

# Split DuoMind
dm_all_ids = list(gold_labels.keys())
random.shuffle(dm_all_ids)
dm_dev_ids  = dm_all_ids[:int(len(dm_all_ids)*0.8)]
dm_test_ids = dm_all_ids[int(len(dm_all_ids)*0.8):]
dm_dev_r    = {q: gold_labels[q] for q in dm_dev_ids}
dm_test_r   = {q: gold_labels[q] for q in dm_test_ids}

print(f"DuoMind dev : {len(dm_dev_ids)}")
print(f"DuoMind test: {len(dm_test_ids)}")

Unique questions: 378
Embedding chunks...


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Embedding answers...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Gold labels: 378
Avg score  : 0.7138
DuoMind dev : 302
DuoMind test: 76


In [33]:
# DuoMind pairs
dm_pairs = []
for question, gold_idx_set in gold_labels.items():
    gold_idx = list(gold_idx_set)[0]
    dm_pairs.append(InputExample(
        texts=[question, chunk_texts[gold_idx]]
    ))
print(f"DuoMind pairs : {len(dm_pairs)}")

# PolicyQA pairs
pqa_pairs = []
for split in ["train.json", "dev.json"]:
    with open(f"/kaggle/working/PolicyQA/data/{split}") as f:
        data = json.load(f)
    for doc in data["data"]:
        for para in doc["paragraphs"]:
            context = para["context"].strip()
            for qa in para["qas"]:
                if qa.get("is_impossible", False): continue
                if not qa["answers"]: continue
                pqa_pairs.append(InputExample(
                    texts=[qa["question"].strip(), context]
                ))

# Sample 5000 PolicyQA
pqa_sample = random.sample(pqa_pairs, min(5000, len(pqa_pairs)))
print(f"PolicyQA pairs: {len(pqa_sample)}")

# Combine
all_pairs = dm_pairs + pqa_sample
random.shuffle(all_pairs)
print(f"Total pairs   : {len(all_pairs)}")

DuoMind pairs : 378
PolicyQA pairs: 5000
Total pairs   : 5378


In [34]:
print("Fine-tuning BGE...")
ft_model = SentenceTransformer("BAAI/bge-base-en-v1.5", device="cuda")

loader  = DataLoader(all_pairs, shuffle=True, batch_size=32)
loss_fn = losses.MultipleNegativesRankingLoss(model=ft_model)

print(f"Pairs          : {len(all_pairs)}")
print(f"Steps/epoch    : {len(loader)}")

ft_model.fit(
    train_objectives  = [(loader, loss_fn)],
    epochs            = 3,
    warmup_steps      = 100,
    output_path       = "/kaggle/working/bge-finetuned",
    show_progress_bar = True
)
print("Fine-tuning complete.")

Fine-tuning BGE...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


Pairs          : 5378
Steps/epoch    : 169


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning complete.


In [35]:
print("Loading fine-tuned BGE...")
ft_bge = SentenceTransformer(
    "/kaggle/working/bge-finetuned", device="cuda"
)

# Re-embed PolicyQA passages
print("Re-embedding PolicyQA passages...")
pqa_emb_ft = ft_bge.encode(
    pqa_passages, batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

# Re-embed DuoMind chunks
print("Re-embedding DuoMind chunks...")
dm_emb_ft = ft_bge.encode(
    chunk_texts, batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print(f"PolicyQA embeddings: {pqa_emb_ft.shape}")
print(f"DuoMind embeddings : {dm_emb_ft.shape}")

Loading fine-tuned BGE...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Re-embedding PolicyQA passages...


Batches:   0%|          | 0/51 [00:00<?, ?it/s]

Re-embedding DuoMind chunks...


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

PolicyQA embeddings: (3202, 768)
DuoMind embeddings : (1476, 768)


In [42]:
# PolicyQA test queries — sample 500
pqa_test_sample = random.sample(pqa_test_ids, 500)

print("Embedding PolicyQA test queries (original BGE)...")
pqa_q_embs_orig = {}
for q in pqa_test_sample:
    pqa_q_embs_orig[q] = embed_query(
        pqa_questions[q], model, tokenizer
    )

print("Embedding PolicyQA test queries (fine-tuned BGE)...")
pqa_q_embs_ft = {}
for q in pqa_test_sample:
    pqa_q_embs_ft[q] = ft_bge.encode(
        [f"Represent this sentence for searching relevant passages: {pqa_questions[q]}"],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]

# DuoMind test queries
print("Embedding DuoMind test queries (original BGE)...")
dm_q_embs_orig = {}
for q in dm_test_ids:
    dm_q_embs_orig[q] = embed_query(q, model, tokenizer)

print("Embedding DuoMind test queries (fine-tuned BGE)...")
dm_q_embs_ft = {}
for q in dm_test_ids:
    dm_q_embs_ft[q] = ft_bge.encode(
        [f"Represent this sentence for searching relevant passages: {q}"],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]

print("All queries embedded.")

Embedding PolicyQA test queries (original BGE)...
Embedding PolicyQA test queries (fine-tuned BGE)...
Embedding DuoMind test queries (original BGE)...
Embedding DuoMind test queries (fine-tuned BGE)...
All queries embedded.


In [43]:
# Original BGE embeddings
print("=== Original BGE Graphs ===")
pqa_graph_orig = LegalDocumentGraph(
    model.encode(pqa_passages, batch_size=64,
                 normalize_embeddings=True,
                 convert_to_numpy=True),
    "mahalanobis"
)
dm_graph_orig = LegalDocumentGraph(
    duomind_embeddings, "mahalanobis"
)

# Fine-tuned BGE embeddings
print("\n=== Fine-tuned BGE Graphs ===")
pqa_graph_ft = LegalDocumentGraph(pqa_emb_ft, "mahalanobis")
dm_graph_ft  = LegalDocumentGraph(dm_emb_ft, "mahalanobis")

print("\nAll graphs built.")

=== Original BGE Graphs ===
Building mahalanobis graph | N=3202 | K=9
  Fitting LedoitWolf...
Edges: 25882
Building mahalanobis graph | N=1476 | K=8
  Fitting LedoitWolf...
Edges: 10853

=== Fine-tuned BGE Graphs ===
Building mahalanobis graph | N=3202 | K=9
  Fitting LedoitWolf...
Edges: 27204
Building mahalanobis graph | N=1476 | K=8
  Fitting LedoitWolf...
Edges: 10979

All graphs built.


In [46]:
# Skip validation entirely — use known best params
# Validated across multiple runs — stable results

best_pqa_orig = {"lambda": 0.5, "alpha": 0.7}
best_pqa_ft   = {"lambda": 0.5, "alpha": 0.7}
best_dm_orig  = {"lambda": 0.7, "alpha": 0.5}
best_dm_ft    = {"lambda": 0.7, "alpha": 0.5}

print("Using validated best params:")
print(f"PolicyQA original : {best_pqa_orig}")
print(f"PolicyQA ft       : {best_pqa_ft}")
print(f"DuoMind original  : {best_dm_orig}")
print(f"DuoMind ft        : {best_dm_ft}")

# Embed only what we need — test queries only
# PolicyQA test sample
print("\nEmbedding PolicyQA test queries...")
pqa_test_sample = random.sample(pqa_test_ids, 500)
pqa_q_embs_orig = {}
pqa_q_embs_ft   = {}
for q in pqa_test_sample:
    pqa_q_embs_orig[q] = embed_query(
        pqa_questions[q], model, tokenizer
    )
    pqa_q_embs_ft[q] = ft_bge.encode(
        [f"Represent this sentence for searching "
         f"relevant passages: {pqa_questions[q]}"],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]
print(f"Done. {len(pqa_test_sample)} queries.")

# DuoMind test queries
print("\nEmbedding DuoMind test queries...")
dm_q_embs_orig = {}
dm_q_embs_ft   = {}
for q in dm_test_ids:
    dm_q_embs_orig[q] = embed_query(q, model, tokenizer)
    dm_q_embs_ft[q]   = ft_bge.encode(
        [f"Represent this sentence for searching "
         f"relevant passages: {q}"],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]
print(f"Done. {len(dm_test_ids)} queries.")

Using validated best params:
PolicyQA original : {'lambda': 0.5, 'alpha': 0.7}
PolicyQA ft       : {'lambda': 0.5, 'alpha': 0.7}
DuoMind original  : {'lambda': 0.7, 'alpha': 0.5}
DuoMind ft        : {'lambda': 0.7, 'alpha': 0.5}

Embedding PolicyQA test queries...
Done. 500 queries.

Embedding DuoMind test queries...
Done. 76 queries.


In [47]:
def run_full_eval(corpus_texts, corpus_emb_orig, corpus_emb_ft,
                  graph_orig, graph_ft,
                  q_embs_orig, q_embs_ft,
                  test_ids, test_r,
                  best_orig, best_ft,
                  model, ft_bge, tokenizer,
                  dataset_name):

    results = {}

    # BM25
    bm25 = BM25Okapi([t.lower().split() for t in corpus_texts])
    bm25_ret = {}
    for q in test_ids:
        scores = bm25.get_scores(q.lower().split())
        ranked = np.argsort(scores)[::-1][:10]
        bm25_ret[q] = [(int(i), float(scores[i])) for i in ranked]
    results["BM25"] = evaluate(
        bm25_ret, test_r, test_ids, label=f"{dataset_name} BM25"
    )

    # BGE original
    bge_orig_ret = {
        q: semantic_retrieve(q_embs_orig[q], corpus_emb_orig)
        for q in test_ids
    }
    results["BGE Original"] = evaluate(
        bge_orig_ret, test_r, test_ids,
        label=f"{dataset_name} BGE Original"
    )

    # BGE fine-tuned
    bge_ft_ret = {
        q: semantic_retrieve(q_embs_ft[q], corpus_emb_ft)
        for q in test_ids
    }
    results["BGE Fine-tuned"] = evaluate(
        bge_ft_ret, test_r, test_ids,
        label=f"{dataset_name} BGE Fine-tuned"
    )

    # SSH-RAG original BGE
    ret_orig = SSHRetriever(
        corpus_texts, corpus_emb_orig, graph_orig,
        model, tokenizer,
        decay_lambda=best_orig["lambda"],
        alpha=best_orig["alpha"]
    )
    ssh_orig_ret = {
        q: ret_orig.retrieve(q_embs_orig[q], q)
        for q in test_ids
    }
    results["SSH-RAG Maha (original)"] = evaluate(
        ssh_orig_ret, test_r, test_ids,
        label=f"{dataset_name} SSH-RAG Maha (original BGE)"
    )

    # SSH-RAG fine-tuned BGE
    ret_ft = SSHRetriever(
        corpus_texts, corpus_emb_ft, graph_ft,
        ft_bge, tokenizer,
        decay_lambda=best_ft["lambda"],
        alpha=best_ft["alpha"]
    )
    ssh_ft_ret = {
        q: ret_ft.retrieve(q_embs_ft[q], q)
        for q in test_ids
    }
    results["SSH-RAG Maha (fine-tuned)"] = evaluate(
        ssh_ft_ret, test_r, test_ids,
        label=f"{dataset_name} SSH-RAG Maha (fine-tuned BGE)"
    )

    # Print table
    print(f"\n{'='*65}")
    print(f"{dataset_name} FINAL RESULTS")
    print(f"{'='*65}")
    print(f"{'Method':<35} {'MAP':>8} {'R@5':>8} {'R@10':>8}")
    print(f"{'-'*65}")
    for method, res in results.items():
        print(f"{method:<35} "
              f"{res['MAP']:>8.4f} "
              f"{res['Recall@5']:>8.4f} "
              f"{res['Recall@10']:>8.4f}")
    print(f"{'='*65}")

    return results


# Run PolicyQA
pqa_test_r_sample = {
    q: pqa_relevance.get(q, set()) for q in pqa_test_sample
}
pqa_results = run_full_eval(
    corpus_texts   = pqa_passages,
    corpus_emb_orig= pqa_orig_embeddings,
    corpus_emb_ft  = pqa_emb_ft,
    graph_orig     = pqa_graph_orig,
    graph_ft       = pqa_graph_ft,
    q_embs_orig    = pqa_q_embs_orig,
    q_embs_ft      = pqa_q_embs_ft,
    test_ids       = pqa_test_sample,
    test_r         = pqa_test_r_sample,
    best_orig      = best_pqa_orig,
    best_ft        = best_pqa_ft,
    model          = model,
    ft_bge         = ft_bge,
    tokenizer      = tokenizer,
    dataset_name   = "PolicyQA"
)

# Run DuoMind
dm_results = run_full_eval(
    corpus_texts   = chunk_texts,
    corpus_emb_orig= duomind_embeddings,
    corpus_emb_ft  = dm_emb_ft,
    graph_orig     = dm_graph_orig,
    graph_ft       = dm_graph_ft,
    q_embs_orig    = dm_q_embs_orig,
    q_embs_ft      = dm_q_embs_ft,
    test_ids       = dm_test_ids,
    test_r         = dm_test_r,
    best_orig      = best_dm_orig,
    best_ft        = best_dm_ft,
    model          = model,
    ft_bge         = ft_bge,
    tokenizer      = tokenizer,
    dataset_name   = "DuoMind"
)


=== PolicyQA BM25 ===
  MAP         : 0.0
  Recall@5    : 0.0
  Recall@10   : 0.0

=== PolicyQA BGE Original ===
  MAP         : 0.1376
  Recall@5    : 0.194
  Recall@10   : 0.26

=== PolicyQA BGE Fine-tuned ===
  MAP         : 0.122
  Recall@5    : 0.184
  Recall@10   : 0.236

=== PolicyQA SSH-RAG Maha (original BGE) ===
  MAP         : 0.1341
  Recall@5    : 0.192
  Recall@10   : 0.26

=== PolicyQA SSH-RAG Maha (fine-tuned BGE) ===
  MAP         : 0.1182
  Recall@5    : 0.184
  Recall@10   : 0.236

PolicyQA FINAL RESULTS
Method                                   MAP      R@5     R@10
-----------------------------------------------------------------
BM25                                  0.0000   0.0000   0.0000
BGE Original                          0.1376   0.1940   0.2600
BGE Fine-tuned                        0.1220   0.1840   0.2360
SSH-RAG Maha (original)               0.1341   0.1920   0.2600
SSH-RAG Maha (fine-tuned)             0.1182   0.1840   0.2360

=== DuoMind BM25 ===
  MA

In [50]:
# Rebuild pqa_emb_orig only
print("Re-embedding PolicyQA passages with original BGE...")
pqa_emb_orig = model.encode(
    pqa_passages,
    batch_size           = 64,
    show_progress_bar    = True,
    normalize_embeddings = True,
    convert_to_numpy     = True
)
print(f"Shape: {pqa_emb_orig.shape}")

Re-embedding PolicyQA passages with original BGE...


Batches:   0%|          | 0/51 [00:00<?, ?it/s]

Shape: (3202, 768)


In [51]:
# Rebuild all four graphs
print("Rebuilding graphs...")

print("1/4 DuoMind original...")
dm_graph_orig  = LegalDocumentGraph(duomind_embeddings, "mahalanobis")

print("2/4 DuoMind fine-tuned...")
graph_maha_ft  = LegalDocumentGraph(dm_emb_ft, "mahalanobis")

print("3/4 PolicyQA original...")
pqa_graph_orig = LegalDocumentGraph(pqa_emb_orig, "mahalanobis")

print("4/4 Done.")

Rebuilding graphs...
1/4 DuoMind original...
Building mahalanobis graph | N=1476 | K=8
  Fitting LedoitWolf...
Edges: 10853
2/4 DuoMind fine-tuned...
Building mahalanobis graph | N=1476 | K=8
  Fitting LedoitWolf...
Edges: 10979
3/4 PolicyQA original...
Building mahalanobis graph | N=3202 | K=9
  Fitting LedoitWolf...
Edges: 25882
4/4 Done.


In [53]:
import types
from collections import defaultdict
import numpy as np

# ── Query-aware propagation function ─────────────────────

def _propagate_query_aware(self, anchor_indices,
                            anchor_scores, q_emb):
    """
    Edge weights amplified by query relevance
    of both endpoint chunks.
    Focuses graph expansion on query-relevant subgraph.
    """
    graph_scores = defaultdict(float)

    for idx, score in zip(anchor_indices, anchor_scores):
        graph_scores[int(idx)] += float(score)

        for neighbor, hops in self.graph.get_neighbors(
            int(idx), self.max_hops
        ).items():
            static_w       = self.graph.get_edge_weight(
                int(idx), neighbor
            )
            q_rel_anchor   = float(
                self.corpus_embeddings[int(idx)] @ q_emb
            )
            q_rel_neighbor = float(
                self.corpus_embeddings[neighbor] @ q_emb
            )
            query_boost    = max(
                0, (q_rel_anchor + q_rel_neighbor) / 2
            )
            combined_w     = static_w * (1 + query_boost)

            graph_scores[neighbor] += (
                float(score) *
                combined_w *
                (self.decay_lambda ** hops)
            )

    return graph_scores


def retrieve_query_aware(self, q_emb, query_text):
    """
    SSH-RAG with query-aware decay propagation.
    """
    bm25_idx, bm25_sc = self._bm25_anchors(query_text)
    sem_idx,  sem_sc  = self._semantic_anchors(q_emb)

    bm25_g = self._propagate_query_aware(
        bm25_idx, bm25_sc, q_emb
    )
    sem_g  = self._propagate_query_aware(
        sem_idx, sem_sc, q_emb
    )

    merged    = {
        c: (bm25_g.get(c,0.0) + sem_g.get(c,0.0)) / 2.0
        for c in set(bm25_g) | set(sem_g)
    }
    cand_list = list(merged.keys())
    cand_embs = self.corpus_embeddings[cand_list]
    sem_final = cand_embs @ q_emb

    g = np.array([merged[c] for c in cand_list])
    if g.max() > 0:
        g = g / g.max()

    s = sem_final
    if s.max() > s.min():
        s = (s - s.min()) / (s.max() - s.min())

    final   = self.alpha * s + (1 - self.alpha) * g
    top_idx = np.argsort(final)[::-1][:self.top_k]

    return [(cand_list[i], float(final[i])) for i in top_idx]


# ── Create retrievers with query-aware method ─────────────

# DuoMind — fine-tuned BGE + Mahalanobis
ret_dm_ft_qa = SSHRetriever(
    corpus_texts      = chunk_texts,
    corpus_embeddings = dm_emb_ft,
    graph             = graph_maha_ft,
    query_model       = ft_bge,
    query_tokenizer   = tokenizer,
    decay_lambda      = 0.7,
    alpha             = 0.5
)
ret_dm_ft_qa._propagate_query_aware = types.MethodType(
    _propagate_query_aware, ret_dm_ft_qa
)
ret_dm_ft_qa.retrieve_query_aware = types.MethodType(
    retrieve_query_aware, ret_dm_ft_qa
)

# DuoMind — original BGE + Mahalanobis
ret_dm_orig_qa = SSHRetriever(
    corpus_texts      = chunk_texts,
    corpus_embeddings = duomind_embeddings,
    graph             = dm_graph_orig,
    query_model       = model,
    query_tokenizer   = tokenizer,
    decay_lambda      = 0.7,
    alpha             = 0.5
)
ret_dm_orig_qa._propagate_query_aware = types.MethodType(
    _propagate_query_aware, ret_dm_orig_qa
)
ret_dm_orig_qa.retrieve_query_aware = types.MethodType(
    retrieve_query_aware, ret_dm_orig_qa
)

# PolicyQA — original BGE + Mahalanobis
ret_pqa_orig_qa = SSHRetriever(
    corpus_texts      = pqa_passages,
    corpus_embeddings = pqa_emb_orig,
    graph             = pqa_graph_orig,
    query_model       = model,
    query_tokenizer   = tokenizer,
    decay_lambda      = 0.5,
    alpha             = 0.7
)
ret_pqa_orig_qa._propagate_query_aware = types.MethodType(
    _propagate_query_aware, ret_pqa_orig_qa
)
ret_pqa_orig_qa.retrieve_query_aware = types.MethodType(
    retrieve_query_aware, ret_pqa_orig_qa
)

# ── Evaluate all three ────────────────────────────────────

print("Running Query-Aware SSH-RAG...")

# DuoMind fine-tuned
print("\nDuoMind (fine-tuned BGE)...")
dm_qa_ft_ret = {
    q: ret_dm_ft_qa.retrieve_query_aware(
        dm_q_embs_ft[q], q
    )
    for q in dm_test_ids
}
r_dm_qa_ft = evaluate(
    dm_qa_ft_ret, dm_test_r, dm_test_ids,
    label="DuoMind SSH-RAG Query-Aware (fine-tuned BGE)"
)

# DuoMind original
print("\nDuoMind (original BGE)...")
dm_qa_orig_ret = {
    q: ret_dm_orig_qa.retrieve_query_aware(
        dm_q_embs_orig[q], q
    )
    for q in dm_test_ids
}
r_dm_qa_orig = evaluate(
    dm_qa_orig_ret, dm_test_r, dm_test_ids,
    label="DuoMind SSH-RAG Query-Aware (original BGE)"
)

# PolicyQA original
print("\nPolicyQA (original BGE)...")
pqa_qa_orig_ret = {
    q: ret_pqa_orig_qa.retrieve_query_aware(
        pqa_q_embs_orig[q], q
    )
    for q in pqa_test_sample
}
pqa_test_r_sample = {
    q: pqa_relevance.get(q, set())
    for q in pqa_test_sample
}
r_pqa_qa_orig = evaluate(
    pqa_qa_orig_ret, pqa_test_r_sample, pqa_test_sample,
    label="PolicyQA SSH-RAG Query-Aware (original BGE)"
)

# ── Final comparison tables ───────────────────────────────

print("\n" + "="*65)
print("DUOMIND — COMPLETE COMPARISON")
print("="*65)
print(f"{'Method':<40} {'MAP':>8} {'R@5':>8} {'R@10':>8}")
print("-"*65)
for label, res in [
    ("BM25",
     {"MAP":0.2371,"Recall@5":0.3158,"Recall@10":0.3882}),
    ("BGE Original",
     {"MAP":0.3678,"Recall@5":0.4934,"Recall@10":0.6382}),
    ("SSH-RAG Maha (original)",
     {"MAP":0.3733,"Recall@5":0.4803,"Recall@10":0.5921}),
    ("SSH-RAG Query-Aware (original)", r_dm_qa_orig),
    ("BGE Fine-tuned",
     {"MAP":0.4508,"Recall@5":0.5724,"Recall@10":0.7368}),
    ("SSH-RAG Maha (fine-tuned)",
     {"MAP":0.4178,"Recall@5":0.5395,"Recall@10":0.6842}),
    ("SSH-RAG Query-Aware (fine-tuned)", r_dm_qa_ft),
]:
    print(f"{label:<40} "
          f"{res['MAP']:>8.4f} "
          f"{res['Recall@5']:>8.4f} "
          f"{res['Recall@10']:>8.4f}")
print("="*65)

print("\n" + "="*65)
print("POLICYQA — COMPLETE COMPARISON")
print("="*65)
print(f"{'Method':<40} {'MAP':>8} {'R@5':>8} {'R@10':>8}")
print("-"*65)
for label, res in [
    ("BM25",
     {"MAP":0.0000,"Recall@5":0.0000,"Recall@10":0.0000}),
    ("BGE Original",
     {"MAP":0.1376,"Recall@5":0.1940,"Recall@10":0.2600}),
    ("SSH-RAG Maha (original)",
     {"MAP":0.1341,"Recall@5":0.1920,"Recall@10":0.2600}),
    ("SSH-RAG Query-Aware (original)", r_pqa_qa_orig),
    ("BGE Fine-tuned",
     {"MAP":0.1220,"Recall@5":0.1840,"Recall@10":0.2360}),
    ("SSH-RAG Maha (fine-tuned)",
     {"MAP":0.1182,"Recall@5":0.1840,"Recall@10":0.2360}),
]:
    print(f"{label:<40} "
          f"{res['MAP']:>8.4f} "
          f"{res['Recall@5']:>8.4f} "
          f"{res['Recall@10']:>8.4f}")
print("="*65)

Running Query-Aware SSH-RAG...

DuoMind (fine-tuned BGE)...

=== DuoMind SSH-RAG Query-Aware (fine-tuned BGE) ===
  MAP         : 0.4196
  Recall@5    : 0.5461
  Recall@10   : 0.6842

DuoMind (original BGE)...

=== DuoMind SSH-RAG Query-Aware (original BGE) ===
  MAP         : 0.3841
  Recall@5    : 0.4803
  Recall@10   : 0.5921

PolicyQA (original BGE)...

=== PolicyQA SSH-RAG Query-Aware (original BGE) ===
  MAP         : 0.1337
  Recall@5    : 0.194
  Recall@10   : 0.256

DUOMIND — COMPLETE COMPARISON
Method                                        MAP      R@5     R@10
-----------------------------------------------------------------
BM25                                       0.2371   0.3158   0.3882
BGE Original                               0.3678   0.4934   0.6382
SSH-RAG Maha (original)                    0.3733   0.4803   0.5921
SSH-RAG Query-Aware (original)             0.3841   0.4803   0.5921
BGE Fine-tuned                             0.4508   0.5724   0.7368
SSH-RAG Maha (fi

In [52]:
# Verify query embeddings still in memory
print(f"dm_q_embs_ft  : {len(dm_q_embs_ft)} queries")
print(f"dm_q_embs_orig: {len(dm_q_embs_orig)} queries")
print(f"pqa_q_embs_orig:{len(pqa_q_embs_orig)} queries")

dm_q_embs_ft  : 76 queries
dm_q_embs_orig: 76 queries
pqa_q_embs_orig:500 queries


In [54]:
# Ablation: No graph expansion
# alpha=1.0 means final score = 100% semantic, 0% graph
ret_ablation = SSHRetriever(
    corpus_texts      = chunk_texts,
    corpus_embeddings = duomind_embeddings,
    graph             = dm_graph_orig,
    query_model       = model,
    query_tokenizer   = tokenizer,
    decay_lambda      = 0.7,
    alpha             = 1.0  # no graph contribution
)

ablation_ret = {
    q: ret_ablation.retrieve(
        dm_q_embs_orig[q], q
    )
    for q in dm_test_ids
}

r_ablation = evaluate(
    ablation_ret, dm_test_r, dm_test_ids,
    label="SSH-RAG No Graph (alpha=1.0)"
)

print("\nAblation comparison:")
print(f"BGE flat retrieval        : MAP={0.3678:.4f}")
print(f"SSH-RAG no graph (α=1.0) : MAP={r_ablation['MAP']:.4f}")
print(f"SSH-RAG with graph (α=0.5): MAP={0.3733:.4f}")
print(f"SSH-RAG QA-graph (α=0.5) : MAP={0.3841:.4f}")


=== SSH-RAG No Graph (alpha=1.0) ===
  MAP         : 0.3678
  Recall@5    : 0.4934
  Recall@10   : 0.6382

Ablation comparison:
BGE flat retrieval        : MAP=0.3678
SSH-RAG no graph (α=1.0) : MAP=0.3678
SSH-RAG with graph (α=0.5): MAP=0.3733
SSH-RAG QA-graph (α=0.5) : MAP=0.3841


In [56]:
# Replace Mahalanobis graph with Euclidean graph
# Keep everything else identical
ret_euclid = SSHRetriever(
    corpus_texts      = chunk_texts,
    corpus_embeddings = duomind_embeddings,
    graph             = dm_graph_orig_euclid,  # euclidean
    query_model       = model,
    query_tokenizer   = tokenizer,
    decay_lambda      = 0.7,
    alpha             = 0.5
)

NameError: name 'dm_graph_orig_euclid' is not defined

In [57]:
import shutil

# Save fine-tuned BGE
ft_bge.save("/kaggle/working/bge-ft-duomind-final")
print("Fine-tuned BGE saved.")

# Zip for download
shutil.make_archive(
    "/kaggle/working/bge-ft-duomind-final",
    "zip",
    "/kaggle/working/bge-ft-duomind-final"
)
print("Zipped. Download from Kaggle output panel.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned BGE saved.
Zipped. Download from Kaggle output panel.
